# Averaging the window

**Temporal-stage notebook 2 in `notebooks/temporal/`.** `notebooks/temporal/README.md` says what the notebooks in this folder share. `01_last_cycle_fd001.ipynb` explains what a temporal stage receives and returns; read it first.

## What this notebook is for

The stage in this notebook, `mean_window`, hands each sensor's node the average of its 30 readings. Like the newest reading, it keeps the one number per sensor that the paper's Table 4 prints, so the network is the same size. Unlike the newest reading, every one of the 30 readings counts, each with the same weight, one thirtieth.

It is not in the paper. When the newest reading was chosen as the stand-in for the paper's recurrent stage, the average was named as the other choice that keeps Table 4's shape (assumption A-07), and this stage is that other choice.

## What it does not do

- **It does not train anything.** The stage has nothing to learn, and the network's results are read from stored runs.
- **It does not compare this stage with another.** `07_comparison_fd001.ipynb` does that.

## The notebook in one list

| # | Step | What goes in | What comes out |
|---|------|--------------|----------------|
| 1 | **Load the window** | `config/fd001.yaml`, seed 42 | the held-back engine's last window |
| 2 | **Average by hand** | T50's 30 readings | their sum and their average, checked against the stage |
| 3 | **Where the average sits** | the same window, and every window of the engine | how far the average trails the newest reading, and why |
| 4 | **Count the cost** | the network built with this stage | learned numbers in the stage and in the whole network |
| 5 | **Read the stored results** | `reports/` | every stored five-seed run of this stage under this folder's settings |

## 1. Background

### An average describes the middle of the window, not its end

Take three readings that rise by one each cycle: 1, 2, 3. Their average is (1 + 2 + 3) / 3 = 2, the middle reading. The newest reading is 3, one cycle after the middle. So on a series that rises steadily, the average is the value the series had at the window's middle, and it trails the newest reading by the rise per cycle times the number of cycles from the middle to the end.

For 30 readings at positions 1 to 30, the middle is position 15.5, the average of the positions, and the newest reading is at position 30, 14.5 cycles later. If the readings lay exactly on a straight line rising by `b` per cycle,

```
newest reading − average  =  14.5 × b
```

where `b` is the line's rise per cycle, in the readings' units. Real readings scatter around any line, so on a real window this holds only roughly; step 3 measures by how much.

That is the trade this stage makes against the newest reading. It is steadier, because one unusual reading moves it by only a thirtieth of its size. But on a sensor that is changing, it describes where the sensor was about 14.5 cycles before the moment being predicted.

**Do not confuse** the average of a window with a smoothed reading. The readings were already smoothed before the windows were cut (assumption A-04), by a filter that gives the newest reading the most weight. The average weighs all 30 equally.

## 2. Setup

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# The project's top folder is the nearest one above this notebook that holds
# pyproject.toml. Adding its src/ folder to Python's search path makes the
# package importable from a clone that was never installed.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import torch
except ImportError as missing:
    raise ImportError(
        "This notebook builds the network's temporal stage, which needs PyTorch, and PyTorch is "
        'not installed. Install the project with  pip install -e ".[model]"'
    ) from missing

from thesis_gnn_rul_xai.config import load_config
from thesis_gnn_rul_xai.data.pipeline import run_preprocessing
from thesis_gnn_rul_xai.models import ModelConfig, build_model
from thesis_gnn_rul_xai.models.temporal import MeanWindow
from thesis_gnn_rul_xai.training import TrainingConfig, find_runs
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging

configure_logging(logging.ERROR)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", None)    # a run's folder name is printed whole
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch     : {torch.__version__}")

## 3. The window

### Step 1 — Load the window

**What.** Rebuild the data from `config/fd001.yaml`, take the held-back engine with the most windows, and take its last window. Print the 30 readings of T50, the temperature at the outlet of the low-pressure turbine, the sensor every notebook in this folder follows.

**Why.** Every notebook here works on this same window, so a difference between two notebooks is a difference between two stages. `notebooks/temporal/README.md` says why this engine, this window and this sensor, and what the units of a reading are.

**Check.** The window must be 14 sensors by 30 cycles, come from an engine the training never saw, and end at the cycle the engine failed.

In [ ]:
config = load_config(PROJECT_ROOT / "config" / "fd001.yaml")
prepared = run_preprocessing(config)

sensors = list(prepared.sensor_symbols)

# The held-back engine with the most windows, and its last window.
held_back = prepared.validation
units, counts = np.unique(held_back.units, return_counts=True)
ENGINE = int(units[np.argmax(counts)])
engine_rows = np.flatnonzero(held_back.units == ENGINE)
engine_rows = engine_rows[np.argsort(held_back.end_cycles[engine_rows])]

SENSOR = "T50"
s = sensors.index(SENSOR)
window = held_back.features[engine_rows[-1]]             # (14 sensors, 30 cycles)
end_cycle = int(held_back.end_cycles[engine_rows[-1]])
cycles = np.arange(end_cycle - window.shape[1] + 1, end_cycle + 1)
readings = window[s]                                     # T50's 30 readings, oldest first

print(f"Engine {ENGINE}: {len(engine_rows)} windows, held back from training under seed {config.seed}.")
print(f"Worked window: cycles {cycles[0]} to {cycles[-1]}, shape {window.shape}, "
      f"remaining life at its end {held_back.targets[engine_rows[-1]]:.0f} cycles.\n")
display(pd.DataFrame([readings], index=[f"{SENSOR} reading"],
                     columns=pd.MultiIndex.from_arrays([range(1, 31), cycles],
                                                       names=["position", "engine cycle"])).round(6))

window_checks = {
    "the window is 14 sensors by 30 cycles": window.shape == (14, 30),
    "its engine is not a training engine": ENGINE not in set(prepared.train.units),
    "it ends at the cycle the engine failed": float(held_back.targets[engine_rows[-1]]) == 0.0,
}
assert all(window_checks.values()), window_checks
print("OK  " + "; ".join(window_checks))

## 4. The stage, step by step

### Step 2 — The average, by hand

**What.** Add T50's 30 readings one at a time, printing the first three additions and the last, divide the total by 30, and compare with what `MeanWindow(30)` hands on.

**Why.** An average is simple, and that is why it is worth seeing once with real numbers: the stage's output is this sum and this division and nothing else.

**Check.** The hand result must equal the stage's output for T50, to within the rounding of 32-bit numbers, and the stage's output for every sensor must equal that sensor's average.

In [ ]:
running = 0.0
for position, reading in enumerate(readings.astype(np.float64), start=1):
    running += reading
    if position <= 3 or position == 30:
        print(f"after reading {position:>2} (cycle {cycles[position - 1]}): "
              f"{running - reading:9.6f} + {reading:9.6f} = {running:9.6f}")
    elif position == 4:
        print("   ...")
by_hand = running / 30
print(f"\naverage = {running:.6f} / 30 = {by_hand:.6f}")

stage = MeanWindow(window_size=30)
with torch.no_grad():
    handed_on = stage(torch.from_numpy(window[None]))      # (1, 14, 1)
print(f"MeanWindow hands T50's node {float(handed_on[0, s, 0]):.6f}\n")
display(pd.DataFrame({"handed on": handed_on[0, :, 0].numpy()}, index=sensors).T.round(6))

hand_matches_stage = bool(abs(by_hand - float(handed_on[0, s, 0])) < 1e-5)
stage_is_the_average = bool(np.allclose(handed_on[0, :, 0].numpy(), window.mean(axis=1), atol=1e-6))
assert hand_matches_stage and stage_is_the_average
print("OK  the hand sum matches the stage, and every sensor's number is its average.")

### Step 3 — Where the average sits

**What.** On the worked window, set T50's average beside its newest reading, and beside 14.5 times the rise per cycle of the straight line that fits the 30 readings best. Then follow the gap between the newest reading and the average through every window of the engine's life.

**Why.** Section 1 said the average trails the newest reading by about 14.5 cycles' worth of change. This checks it on real readings, and shows when in an engine's life the two numbers part.

In [ ]:
positions = np.arange(1, 31)
rise_per_cycle, intercept = np.polyfit(positions, readings.astype(np.float64), deg=1)
newest, average = float(readings[-1]), float(readings.mean())
line_at_middle, line_at_end = intercept + rise_per_cycle * 15.5, intercept + rise_per_cycle * 30
print(f"newest reading                         {newest:9.6f}")
print(f"average of the 30                      {average:9.6f}")
print(f"newest minus average                   {newest - average:9.6f}\n")
print(f"best line: rises {rise_per_cycle:.6f} per cycle")
print(f"  its value at the middle, 15.5        {line_at_middle:9.6f}   (the average, as for every best line)")
print(f"  its value at position 30             {line_at_end:9.6f}")
print(f"  14.5 × rise per cycle                {14.5 * rise_per_cycle:9.6f}")
print(f"newest reading minus the line at 30    {newest - line_at_end:9.6f}")
print(f"the two parts added                    {14.5 * rise_per_cycle + newest - line_at_end:9.6f}")

life = held_back.features[engine_rows, s, :]              # (240 windows, 30 cycles)
life_newest, life_average = life[:, -1], life.mean(axis=1)
life_gap = life_newest - life_average
thirds = np.array_split(np.arange(len(engine_rows)), 3)
print("\nnewest minus average over the engine's life, averaged over each third of its windows: "
      + ", ".join(f"{life_gap[rows].mean():+.3f}" for rows in thirds))

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
left.plot(positions, readings, "o-", color="0.55", markersize=3, label=f"{SENSOR} readings")
left.axhline(average, color="C0", label=f"average, {average:.3f}")
left.plot([30], [newest], "o", color="C3", markersize=8, label=f"newest, {newest:.3f}")
left.axvline(15.5, color="C0", linestyle=":", linewidth=1)
left.set(title=f"Engine {ENGINE}, cycles {cycles[0]} to {cycles[-1]}",
         xlabel="position in the window (dotted: the middle, 15.5)", ylabel=f"{SENSOR} reading")
left.legend()
ends = held_back.end_cycles[engine_rows]
right.plot(ends, life_newest, color="C3", linewidth=1, label="newest reading")
right.plot(ends, life_average, color="C0", linewidth=1.5, label="average of the window")
right.set(title=f"Every window of engine {ENGINE}", xlabel="cycle at which the window ends",
          ylabel=f"{SENSOR} reading")
right.legend()
figure.tight_layout()
plt.show()

#### Reading the figure

**On the worked window** the average, 1.838, trails the newest reading, 2.740, by 0.902. The best straight line accounts for 0.530 of that. It rises 0.036550 per cycle, and it passes through the average at the window's middle — every best line does, which is a property of how the line is fitted, not of these readings — so 14.5 cycles later it has reached 1.838 + 0.530 = 2.368. The other 0.372 is the newest reading lying above the line: T50 jumped over the last four cycles, from 1.867 at cycle 265 to between 2.48 and 2.83 at cycles 266 to 269.

**Over the engine's life** the two lines on the right run close together while T50 changes slowly, and part as it climbs. Averaged over each third of the engine's 240 windows, the newest reading sits above the average by 0.066, 0.138 and 0.436: the closer the engine is to failure, the further behind its sensor the average is. That is what this stage gives up in exchange for steadiness. What giving it up cost this network is in the stored runs below, and set beside the other stages in `07_comparison_fd001.ipynb`.

### Step 4 — What it costs

**What.** Build the whole network with this stage and count its learned numbers, as `01_last_cycle_fd001.ipynb` did for the newest reading.

**Why.** The average keeps Table 4's one number per sensor, so it should cost exactly what the newest reading costs. This confirms it.

**Check.** No learned numbers in the stage, `W(1)` of shape `(1, 32)`, and 61,729 in the whole network.

In [ ]:
TABLE_4_NAMES = {"graph_layers.0": "GCN-1", "graph_layers.1": "GCN-2",
                 "graph_layers.2": "GCN-3", "fc1": "FC-1", "fc2": "FC-2"}


def parameter_table(network):
    """Learned numbers in each part of a network, the temporal stage first."""
    parts = {"temporal stage": 0}
    for name, parameter in network.named_parameters():
        part = "temporal stage" if name.startswith("temporal.") else TABLE_4_NAMES[name.rsplit(".", 1)[0]]
        parts[part] = parts.get(part, 0) + parameter.numel()
    parts["whole network"] = sum(parts.values())
    return pd.DataFrame({"learned numbers": parts})


network = build_model(prepared.adjacency_normalized, ModelConfig(temporal="mean_window"),
                      window_size=config.window_size, rul_max=config.rul_max)
costs = parameter_table(network)
display(costs.T)
cost_checks = {
    "the stage learns nothing": int(costs.loc["temporal stage", "learned numbers"]) == 0,
    "W(1) is (1, 32)": tuple(network.graph_layers[0].weight.shape) == (1, 32),
    "the whole network holds 61,729": int(costs.loc["whole network", "learned numbers"]) == 61_729,
}
assert all(cost_checks.values()), cost_checks
print("OK  " + "; ".join(cost_checks))

### Step 5 — The stored results of this stage

**What.** Every stored five-seed run that used this stage under this folder's settings. Runs that saved only part of their settings — the oldest saved nothing about their training but the learning-rate path — are listed with what they did not save, and their seeds are set beside those of the runs that saved everything.

**Why.** As for every stage, an error in cycles belongs to the whole trained network and its settings, so the cell prints the settings beside the runs.

In [ ]:
STUDY = PROJECT_ROOT / "reports" / "sensitivity_fd65a3b-dirty_20260929T101059Z" / "runs" / "FD001"
DEFAULT_TRAINING = TrainingConfig()
TRAINING_FIELDS = ("epochs", "batch_size", "learning_rate", "final_learning_rate",
                   "early_stopping_patience", "lr_schedule")
WANTED = [ModelConfig(temporal="mean_window")]


def training_settings(run) -> str:
    """'default' or 'other' from the training settings a run saved; 'not saved' if it saved only the learning-rate path."""
    recorded = run.summary.get("training")
    if recorded is None:
        return "not saved" if run.summary.get("lr_schedule") == DEFAULT_TRAINING.lr_schedule else "other"
    default = all(recorded.get(field) == getattr(DEFAULT_TRAINING, field) for field in TRAINING_FIELDS)
    return "default" if default else "other"


def not_saved(run) -> str:
    """The settings a run did not save, or a dash if it saved them all."""
    missing = [] if run.summary.get("training") else ["training settings"]
    if not (run.path / "model_config.yaml").is_file():
        missing.append("network sizes" if "temporal_stage" in run.summary else "network sizes, stage")
    return ", ".join(missing) or "—"


def machine(run) -> str:
    """The device a run's first seed asked for, and the PyTorch build the run recorded."""
    saved = run.path / f"seed_{run.seeds[0]}" / "training_config.yaml"
    device = yaml.safe_load(saved.read_text(encoding="utf-8")).get("device") if saved.is_file() else None
    build = ((run.summary.get("environment") or {}).get("packages") or {}).get("torch")
    return f"{device or 'not recorded'}, PyTorch {build or 'not recorded'}"


stored = [
    run
    for root in (None, STUDY)
    for run in find_runs(root=root, preprocessing=config, seeds=5, split="varies_with_seed",
                         early_stopping=True)
    if run.model_config in WANTED and training_settings(run) != "other"
]
print("Settings every run below shares, where it saved them:")
print("  preprocessing:", {k: v for k, v in config.to_dict().items() if k != "notes"})
print("  training:     ", {field: getattr(DEFAULT_TRAINING, field) for field in TRAINING_FIELDS},
      "and the pass that scored best on the held-back engines restored")
for model_config in WANTED:
    print("  network:      ", model_config.to_dict())
print()
if stored:
    display(pd.DataFrame([{
        "run, under reports/": run.path.resolve().relative_to((PROJECT_ROOT / "reports").resolve()).as_posix(),
        "commit": run.summary["commit"],
        **({"one network for all sensors": run.model_config.shared_across_sensors}
           if len(WANTED) > 1 else {}),
        "device asked for, PyTorch build": machine(run),
        "not saved by the run": not_saved(run),
        "seeds": ", ".join(map(str, run.seeds)),
        "test RMSE": f"{run.results['rmse'].mean():.3f} ± {run.results['rmse'].std():.3f}",
        "held-back RMSE": f"{run.results['validation_rmse'].mean():.3f} ± "
                          f"{run.results['validation_rmse'].std():.3f}",
    } for run in stored]).set_index("run, under reports/"))
else:
    print("reports/ holds no five-seed run of this stage under these settings. Make one with\n\n"
          "    python scripts/train.py --config config/fd001.yaml --temporal mean_window --seeds 5\n")

# A run that saved less is set beside the runs that saved everything, seed by seed.
complete = [run for run in stored if not_saved(run) == "—"]
repeats = {}
for run in stored:
    if not_saved(run) == "—":
        continue
    mine = run.results.set_index("seed").sort_index()[["rmse", "validation_rmse"]].to_numpy()
    repeats[run.name] = [
        other.summary["commit"] for other in complete
        if np.array_equal(other.results.set_index("seed").sort_index()[["rmse", "validation_rmse"]].to_numpy(), mine)
    ]
    unsaid = not_saved(run).split(", ")
    unsaid = unsaid[0] if len(unsaid) == 1 else ", ".join(unsaid[:-1]) + " or " + unsaid[-1]
    twins = repeats[run.name]
    print(f"The run at {run.summary['commit']} saved no {unsaid}; its five seeds give the same test and held-back "
          f"RMSE, to every stored digit, as the run{'s' if len(twins) > 1 else ''} at {', '.join(twins) or 'no commit'}.")
unsaved_runs_repeat = all(repeats.values())

**The only stored measurement of this stage** is the sensitivity study's run of 2026-09-29, at `fd65a3b-dirty`, under the settings printed above (finding F-19).

**Which published row it may be set beside: none.** The paper does not say what its variant without the recurrent stage hands the graph, and this project's stand-in for that variant is the newest reading (assumption A-07), not the average.

## Final verification

In [ ]:
checks = [
    ("The window is the folder's window", all(window_checks.values()), f"engine {ENGINE}, cycle {end_cycle}"),
    ("The hand sum matches the stage", hand_matches_stage, f"{by_hand:.6f}"),
    ("Every sensor's number is its average", stage_is_the_average, "14 sensors"),
    ("The cost is Table 4's network", all(cost_checks.values()), "0 in the stage, 61,729 in all"),
    ("Each run that saved less repeats one that saved all", unsaved_runs_repeat, f"{len(repeats)} such run(s)"),
]
width = max(len(name) for name, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 40))
for name, passed, detail in checks:
    print(f"{name:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")
print("-" * (width + 40))
failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")
print(f"\nStored five-seed runs of this stage under this folder's settings: {len(stored)}; the step on stored results lists them, or prints the command that makes one.")
print(f"Measured, not checked: on the worked window the average trails the newest reading by {newest - average:.3f}, against {14.5 * rise_per_cycle:.3f} for a straight line.")

## Summary

### What this notebook produced

- **The average of T50's 30 readings**, added up by hand and checked against the stage.
- **Where it sits**: on a sensor that is changing, about 14.5 cycles' worth of change behind the newest reading, measured on the worked window and over the engine's life.
- **Its cost**: the same as the newest reading's, no learned numbers in the stage and 61,729 in the network.
- **Its stored results**, with their commit and settings.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| What a sensor's node carries in this stage | the average of its 30 readings | A-07 |
| Which window and which sensor are shown | the held-back engine's last window, T50 | `notebooks/temporal/README.md` |

### References

- Wen, J.-R., et al. (2026). Explainable spatiotemporal graph neutral network with dynamic SHAP enhancement for aeroengine RUL prediction. *Aerospace Science and Technology*, 168, 110875. Table 4.